# Batch convert ORD Parquet datasets to CSV

Edit **INPUT_FOLDER** and **OUTPUT_FOLDER**, then run the cells from top to bottom.
Each ORD Parquet file produces one CSV with **the same name as the input file**.
For example, `Newman-Stonebraker_3b_Suzuki_Ni-ligand-screens.parquet` becomes **`Newman-Stonebraker_3b_Suzuki_Ni-ligand-screens.csv`**.

This reads the actual Parquet files and decodes their Protobuf reaction records with ORD's official schema.
It requires **Python 3.11 or newer**. The package installation cell uses the current notebook kernel.


## Set your folder paths

The defaults below use your Windows Downloads directory. Replace them with your actual paths.
Set `RECURSIVE = True` if the Parquet files are spread across subfolders, such as ORD's `data/3b/` structure.
Existing CSVs are skipped unless you set `OVERWRITE = True`.


In [ ]:
from pathlib import Path

INPUT_FOLDER = Path(r"C:\Users\koand\OneDrive\Documents\GitHub\DOPE-MURI\ORD_files\ORD_datasets_parquet")
OUTPUT_FOLDER = Path(r"C:\Users\koand\OneDrive\Documents\GitHub\DOPE-MURI\ORD_files\ORD_datasets_csv")

RECURSIVE = False
OVERWRITE = False
BATCH_SIZE = 1024


## Conversion functions

- Dataset IDs come from `ord.dataset_id` in the Parquet metadata. If absent, the notebook checks the filename for a full ORD dataset ID.
- One reaction becomes one row. Common chemical and condition summaries appear first; `source.*` columns retain the populated nested source fields.
- Repeated entries use one-based indices, such as `source.outcomes[1].products[1]`. Enum codes become readable names; binary data becomes Base64 text.
- Missing values remain blank; recorded zero and False values remain intact. Yield percentages are **0–100**, not fractions.
- `primary_*` summaries use the first outcome and its first desired product (or first product if none is marked desired). All additional outcomes, products, measurements, and quantities remain in `source.*` columns.
- ORD can encode a phosphine ligand as a CATALYST; `catalyst_names` therefore includes all components with that role.
- Two streaming passes discover all columns and write the CSV without loading the entire dataset into RAM. CSVs use UTF-8 with a BOM for Excel.
- If two input files would receive the same CSV filename (possible with `RECURSIVE = True`), both are reported as errors. Failed conversions do not publish a partial CSV.

These functions expect **ORD Parquet files**, with `reaction_id` and binary `reaction` columns. Other Parquet layouts are reported as errors.


In [3]:
from pathlib import Path
from collections import Counter
import base64
import csv
import os
import re
import tempfile

import pandas as pd
import pyarrow.parquet as pq
from google.protobuf.descriptor import FieldDescriptor
from ord_schema.proto import reaction_pb2


def message_dict(message):
    """Keep only recorded fields, including explicit zero and False values."""
    def value_for(field, value):
        if field.type == FieldDescriptor.TYPE_MESSAGE:
            return message_dict(value)
        if field.type == FieldDescriptor.TYPE_ENUM:
            enum_value = field.enum_type.values_by_number.get(value)
            return enum_value.name if enum_value else value
        if field.type == FieldDescriptor.TYPE_BYTES:
            return base64.b64encode(value).decode('ascii')
        return value

    result = {}
    for field, value in message.ListFields():
        is_map = (field.type == FieldDescriptor.TYPE_MESSAGE
                  and field.message_type.GetOptions().map_entry)
        if is_map:
            value_field = field.message_type.fields_by_name['value']
            result[field.name] = {
                str(key): value_for(value_field, value[key])
                for key in sorted(value)
            }
        elif field.label == FieldDescriptor.LABEL_REPEATED:
            result[field.name] = [value_for(field, item) for item in value]
        else:
            result[field.name] = value_for(field, value)
    return result


def flatten(value, prefix='', result=None):
    """Nested maps use dot paths; repeated fields use 1-based [indices]."""
    if result is None:
        result = {}
    if isinstance(value, dict):
        for key, item in value.items():
            flatten(item, f'{prefix}.{key}' if prefix else key, result)
    elif isinstance(value, list):
        for index, item in enumerate(value, 1):
            flatten(item, f'{prefix}[{index}]', result)
    else:
        if prefix in result:
            raise ValueError(f'Flattened column collision: {prefix}')
        result[prefix] = value
    return result


def identifiers(compound, identifier_type):
    return [item['value'] for item in compound.get('identifiers', [])
            if item.get('type') == identifier_type and 'value' in item]


def unique_join(values):
    return '; '.join(dict.fromkeys(values))


def csv_row(reaction, metadata):
    """Convenient summaries first; all populated source fields follow."""
    components = [component
                  for inp in reaction.get('inputs', {}).values()
                  for component in inp.get('components', [])]
    row = {
        'dataset_id': metadata['dataset_id'],
        'dataset_name': metadata['dataset_name'],
        'reaction_id': reaction.get('reaction_id', ''),
    }
    for role in ('REACTANT', 'REAGENT', 'CATALYST', 'SOLVENT'):
        selected = [c for c in components if c.get('reaction_role') == role]
        row[role.lower() + '_names'] = unique_join(
            name for c in selected for name in identifiers(c, 'NAME'))
        row[role.lower() + '_smiles'] = unique_join(
            smiles for c in selected for smiles in identifiers(c, 'SMILES'))
    temperature = reaction.get('conditions', {}).get('temperature', {}).get('setpoint', {})
    row['temperature'] = temperature.get('value')
    row['temperature_unit'] = temperature.get('units')
    outcomes = reaction.get('outcomes', [])
    outcome = outcomes[0] if outcomes else {}
    time = outcome.get('reaction_time', {})
    row['primary_reaction_time'] = time.get('value')
    row['primary_reaction_time_unit'] = time.get('units')
    products = outcome.get('products', [])
    product = next((p for p in products if p.get('is_desired_product')), products[0] if products else {})
    row['primary_product_smiles'] = unique_join(identifiers(product, 'SMILES'))
    measurements = product.get('measurements', [])
    measurement = next((m for m in measurements if m.get('type') == 'YIELD'), {})
    row['primary_yield_percent'] = measurement.get('percentage', {}).get('value')
    row['primary_yield_analysis'] = measurement.get('analysis_key')
    row['publication_doi'] = reaction.get('provenance', {}).get('doi')
    row.update({'source.' + key: value for key, value in flatten(reaction).items()
                if key != 'reaction_id'})
    return row


def parquet_metadata(path):
    with pq.ParquetFile(path) as parquet_file:
        raw = parquet_file.schema_arrow.metadata or {}
        columns = parquet_file.schema_arrow.names
        if not {'reaction', 'reaction_id'}.issubset(columns):
            raise ValueError('Expected ORD Parquet columns reaction and reaction_id.')
        if raw.get(b'ord.schema_version', b'1') != b'1':
            raise ValueError('Unsupported ORD Parquet schema version.')
        dataset_id = raw.get(b'ord.dataset_id', b'').decode('utf-8').strip()
        if not dataset_id:
            match = re.search(r'ord_dataset-[0-9a-fA-F]{32}', path.stem)
            dataset_id = match.group(0) if match else ''
        if not re.fullmatch(r'ord_dataset-[0-9a-fA-F]{32}', dataset_id):
            raise ValueError('Dataset ID missing or invalid in metadata and filename.')
        return {
            'dataset_id': dataset_id,
            'dataset_name': raw.get(b'ord.name', b'').decode('utf-8'),
            'row_count': parquet_file.metadata.num_rows,
        }


def iter_rows(path, metadata, batch_size):
    with pq.ParquetFile(path) as parquet_file:
        for batch in parquet_file.iter_batches(
                batch_size=batch_size, columns=['reaction_id', 'reaction']):
            for record in batch.to_pylist():
                if record['reaction'] is None:
                    raise ValueError('A reaction blob is missing.')
                reaction = message_dict(reaction_pb2.Reaction.FromString(record['reaction']))
                blob_id = reaction.get('reaction_id', '')
                if blob_id and record['reaction_id'] and blob_id != record['reaction_id']:
                    raise ValueError(f'Reaction ID mismatch: {record["reaction_id"]}')
                reaction['reaction_id'] = record['reaction_id'] or blob_id
                if not reaction['reaction_id']:
                    raise ValueError('A reaction ID is missing.')
                yield csv_row(reaction, metadata)


def convert_one(path, metadata, destination, batch_size):
    # Discover every column in a streaming pass, including fields appearing late.
    # A second streaming pass writes rows without holding a whole dataset in RAM.
    columns = {}
    row_count = 0
    for row in iter_rows(path, metadata, batch_size):
        columns.update(dict.fromkeys(row))
        row_count += 1
    if row_count != metadata['row_count']:
        raise ValueError('Parquet row count did not match decoded reaction count.')
    headers = list(columns) or ['dataset_id', 'dataset_name', 'reaction_id']
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
                mode='w', encoding='utf-8-sig', newline='', delete=False,
                dir=destination.parent, prefix=destination.stem + '_', suffix='.tmp') as handle:
            temporary_path = Path(handle.name)
            writer = csv.DictWriter(handle, fieldnames=headers)
            writer.writeheader()
            written = 0
            for row in iter_rows(path, metadata, batch_size):
                writer.writerow({key: str(value).lower() if isinstance(value, bool) else value
                                 for key, value in row.items()})
                written += 1
        if written != row_count:
            raise ValueError('CSV row count changed between passes.')
        os.replace(temporary_path, destination)
        temporary_path = None
        return written, len(headers)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


def convert_folder(input_folder, output_folder, recursive=False, overwrite=False, batch_size=1024):
    input_folder = Path(input_folder).expanduser()
    output_folder = Path(output_folder).expanduser()
    if not input_folder.is_dir():
        raise FileNotFoundError(f'Input folder does not exist: {input_folder}')
    if batch_size < 1:
        raise ValueError('BATCH_SIZE must be at least 1.')
    files = sorted(path for path in (input_folder.rglob('*') if recursive else input_folder.iterdir())
                   if path.is_file() and path.suffix.lower() == '.parquet')
    if not files:
        raise FileNotFoundError(f'No Parquet files found in: {input_folder}')
    output_folder.mkdir(parents=True, exist_ok=True)
    jobs, results = [], []
    for path in files:
        try:
            metadata = parquet_metadata(path)
            destination = output_folder / (path.stem + '.csv')
            jobs.append((path, metadata, destination))
        except Exception as exc:
            results.append({'input_file': str(path), 'status': 'error', 'details': str(exc)})
    counts = Counter(destination.name.casefold() for _, _, destination in jobs)
    for path, metadata, destination in jobs:
        result = {'input_file': str(path), 'dataset_id': metadata['dataset_id'],
                  'output_file': str(destination), 'rows': None, 'columns': None}
        try:
            if counts[destination.name.casefold()] > 1:
                raise ValueError('Multiple input files share this name; rename one or convert the folders separately.')
            if destination.exists() and not overwrite:
                result.update(status='skipped', details='CSV exists; set OVERWRITE=True to replace.')
            else:
                rows, cols = convert_one(path, metadata, destination, batch_size)
                result.update(status='converted', rows=rows, columns=cols, details='')
        except Exception as exc:
            result.update(status='error', details=str(exc))
        results.append(result)
        print(f'{result["status"]}: {path.name} -> {destination.name}'
              + (f' ({result["details"]})' if result['details'] else ''))
    return pd.DataFrame(results)


## Convert every file and review the results

The results table lists each file as `converted`, `skipped`, or `error`. An error in one input does not stop the other conversions.


In [4]:
results = convert_folder(
    input_folder=INPUT_FOLDER,
    output_folder=OUTPUT_FOLDER,
    recursive=RECURSIVE,
    overwrite=OVERWRITE,
    batch_size=BATCH_SIZE,
)

print("\nConversion summary:")
print(results["status"].value_counts().to_string())
display(results)


converted: Ahneman_46_Buchwald-Hartwig_fragment-additives.parquet -> 1ef949c3.csv
converted: Buitrago-Santanilla_7d_cross-coupling_Experiment-2.parquet -> 9b052746.csv
converted: Buitrago-Santanilla_cb_Buchwald-Hartwig_model-screen_triplicates.parquet -> 2549c70d.csv
converted: Gesmundo_17_N-alkylation_Boc-deprotection.parquet -> d203b79b.csv
converted: Gesmundo_2b_reductive-amination_optimization-and-library.parquet -> af6dd1bc.csv
converted: Gesmundo_3b_Suzuki_12x10-informer-library.parquet -> 206a3ac8.csv
converted: King-Smith_d9_mixed-reactions_HiTEA.parquet -> e7048086.csv
converted: Newman-Stonebraker_3b_Suzuki_Ni-ligand-screens.parquet -> c5e3aa72.csv
converted: Perera_68_Suzuki_nanoscale-flow.parquet -> ae58b203.csv
converted: Reizman_20_Suzuki_benzofuran-boronic-acid.parquet -> 8437e929.csv
converted: Reizman_2b_Suzuki_2-chloropyridine_N-Boc-pyrrole-boronic-acid.parquet -> dee05902.csv
converted: Reizman_31_Suzuki_3-bromoquinoline_isoxazole-Bpin.parquet -> 82da4517.csv
convert

,input_file,dataset_id,output_file,rows,columns,status,details
0,C:\Users\koand\OneDrive\Documents\GitHub\DOPE-...,ord_dataset-46ff9a32d9e04016b9380b1b1ef949c3,ORD_files\ORD_datasets_csv\1ef949c3.csv,4312,189,converted,
1,C:\Users\koand\OneDrive\Documents\GitHub\DOPE-...,ord_dataset-7d8f5fd922d4497d91cb81489b052746,ORD_files\ORD_datasets_csv\9b052746.csv,1536,278,converted,
2,C:\Users\koand\OneDrive\Documents\GitHub\DOPE-...,ord_dataset-cbcc4048add7468e850b6ec42549c70d,ORD_files\ORD_datasets_csv\2549c70d.csv,288,173,converted,
3,C:\Users\koand\OneDrive\Documents\GitHub\DOPE-...,ord_dataset-172039a759a440219a68af62d203b79b,ORD_files\ORD_datasets_csv\d203b79b.csv,104,321,converted,
4,C:\Users\koand\OneDrive\Documents\GitHub\DOPE-...,ord_dataset-2be11f57f3304e678ea8469baf6dd1bc,ORD_files\ORD_datasets_csv\af6dd1bc.csv,1152,359,converted,
5,C:\Users\koand\OneDrive\Documents\GitHub\DOPE-...,ord_dataset-3b8a2ef300e145468579027f206a3ac8,ORD_files\ORD_datasets_csv\206a3ac8.csv,1440,261,converted,
6,C:\Users\koand\OneDrive\Documents\GitHub\DOPE-...,ord_dataset-d92976309c3a48a3a64a4cf5e7048086,ORD_files\ORD_datasets_csv\e7048086.csv,39347,245,converted,
7,C:\Users\koand\OneDrive\Documents\GitHub\DOPE-...,ord_dataset-3b5db90e337942ea886b8f5bc5e3aa72,ORD_files\ORD_datasets_csv\c5e3aa72.csv,450,169,converted,
8,C:\Users\koand\OneDrive\Documents\GitHub\DOPE-...,ord_dataset-68cb8b4b2b384e3d85b5b1efae58b203,ORD_files\ORD_datasets_csv\ae58b203.csv,5760,160,converted,
9,C:\Users\koand\OneDrive\Documents\GitHub\DOPE-...,ord_dataset-2038a3c967db4a32a8fbce288437e929,ORD_files\ORD_datasets_csv\8437e929.csv,96,176,converted,


Sources: [ORD data repository](https://github.com/open-reaction-database/ord-data),
[ORD schema package](https://pypi.org/project/ord-schema/), and
[ORD Parquet serialization](https://github.com/open-reaction-database/ord-schema/blob/main/ord_schema/parquet.py).

The converter was tested with the 450-reaction nickel Suzuki–Miyaura Parquet dataset,
including batching, missing fields, zero yields, dataset ID naming, existing-file skips, and filename collisions.
